# MTCA-1 Stage 5: Pilot Execution

**Study identifier:** MTCA-1
**Stage:** 5 of 10 — Pilot validation
**Inputs:** Stage 3 design (frozen at SHA `6380e711bbaed3af`)
**Outputs:** 40 council responses (1 unit × 8 frames × 5 models) + validation report

## Purpose

Validate end-to-end wiring before committing to Stage 6's full 2,000-call execution. The pilot exercises every API client, every frame template, every storage path, and every parser on a small subset where failures are cheap.

## Pilot scope

- **One unit** from the Stage 3 sample (configurable; defaults to a C8 synthetic unit for clear signal)
- **All 8 frames** applied to that unit
- **All 5 council models** queried per frame
- **Total: 40 API calls**, ~$0.40-$1 in cost, 10-20 minutes wall-clock

## Success criteria

Pilot passes if:
1. All 5 API clients authenticate successfully
2. All 40 calls return responses (no timeouts, no auth failures)
3. ≥90% of responses parse as valid JSON with the 5 expected keys
4. Responses show meaningful variation across frames (sanity check, not statistical)
5. Storage structure handles all response shapes cleanly

If any criterion fails, fix before Stage 6.

## Section 1: Environment and Stage 3 design load

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive, userdata
    try:
        drive.mount('/content/drive')
    except Exception:
        pass
    PROJECT_ROOT = '/content/drive/MyDrive/MTCA-1'
else:
    PROJECT_ROOT = './MTCA-1'
    userdata = None

import os
import json
import re
import time
import hashlib
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict

# Output dirs
for subdir in ['stage5', 'stage5/responses', 'stage5/logs']:
    os.makedirs(f'{PROJECT_ROOT}/{subdir}', exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Load Stage 3 design + execution plan
stage3_dir = f'{PROJECT_ROOT}/stage3'
design_files = [f for f in os.listdir(stage3_dir) if f.startswith('stage3_design_')]
if not design_files:
    raise FileNotFoundError('No Stage 3 design found. Run Stage 3 first.')

design_path = f'{stage3_dir}/{design_files[0]}'
with open(design_path, 'r', encoding='utf-8') as f:
    stage3_design = json.load(f)

exec_files = [f for f in os.listdir(stage3_dir) if f.startswith('execution_plan_')]
exec_path = f'{stage3_dir}/{exec_files[0]}'
with open(exec_path, 'r', encoding='utf-8') as f:
    execution_plan_doc = json.load(f)

EXECUTION_PLAN = execution_plan_doc['calls']
DESIGN_SHA = stage3_design['design_sha256']

print(f'Loaded Stage 3 design')
print(f'Design SHA256: {DESIGN_SHA}')
print(f'Total planned calls: {len(EXECUTION_PLAN)}')
print(f'Frames: {len(stage3_design["frames"])}')
print(f'Council models: {[m["model_id"] for m in stage3_design["council_models"]]}')

## Section 2: Install and import provider SDKs

In [ ]:
# Install all required SDKs (no-op if already installed)
!pip install -q anthropic openai google-generativeai

In [ ]:
from anthropic import Anthropic
from openai import OpenAI
import google.generativeai as genai

# Pull API keys from Colab Secrets (or env vars locally)
def get_key(name):
    if IN_COLAB and userdata is not None:
        try:
            return userdata.get(name)
        except Exception:
            pass
    return os.environ.get(name)

API_KEYS = {
    'anthropic': get_key('ANTHROPIC_API_KEY'),
    'openai': get_key('OPENAI_API_KEY'),
    'google': get_key('GOOGLE_API_KEY'),
    'xai': get_key('XAI_API_KEY'),
    'deepseek': get_key('DEEPSEEK_API_KEY'),
}

# Verify all keys present
missing = [k for k, v in API_KEYS.items() if not v]
if missing:
    raise RuntimeError(f'Missing API keys: {missing}. Add to Colab Secrets.')

print('All five API keys loaded.')
for provider in API_KEYS:
    key = API_KEYS[provider]
    print(f'  {provider}: {key[:8]}... ({len(key)} chars)')

In [ ]:
# Initialize clients
anthropic_client = Anthropic(api_key=API_KEYS['anthropic'])
openai_client = OpenAI(api_key=API_KEYS['openai'])
xai_client = OpenAI(api_key=API_KEYS['xai'], base_url='https://api.x.ai/v1')
deepseek_client = OpenAI(api_key=API_KEYS['deepseek'], base_url='https://api.deepseek.com')

genai.configure(api_key=API_KEYS['google'])

print('All five API clients initialized.')

## Section 3: Response data structure

In [ ]:
@dataclass
class CouncilResponse:
    """A single response from one model on one (unit, frame) pair."""
    # Provenance from the execution plan
    call_id: str
    unit_id: str
    text_id: str
    frame_id: str
    frame_version: str
    prompt_sha256: str
    model_id: str
    provider: str
    api_model: str
    
    # Execution metadata
    timestamp: str
    wall_clock_seconds: float
    
    # Response
    raw_response: str  # exactly what the model returned
    parsed_json: dict = field(default_factory=dict)  # parsed if successful
    parse_succeeded: bool = False
    parse_error: str = ''
    
    # Token usage (for cost tracking)
    input_tokens: int = 0
    output_tokens: int = 0
    
    # Error tracking
    error: str = ''
    retry_count: int = 0


def extract_json_from_response(raw: str) -> tuple:
    """Extract and parse JSON from a model response.
    
    Handles markdown code fences, leading/trailing whitespace, and common variations.
    Returns (parsed_dict, success_bool, error_msg).
    """
    cleaned = raw.strip()
    
    # Strip markdown code fences if present
    if cleaned.startswith('```'):
        cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
        cleaned = re.sub(r'\s*```$', '', cleaned)
    
    # Some models add a preamble - try to find the JSON block
    json_start = cleaned.find('{')
    json_end = cleaned.rfind('}')
    
    if json_start == -1 or json_end == -1:
        return {}, False, 'No JSON object found in response'
    
    json_candidate = cleaned[json_start:json_end + 1]
    
    try:
        parsed = json.loads(json_candidate)
        # Validate expected keys present
        expected_keys = {'communicative_function', 'implicit_claims',
                        'coherence_assessment', 'conditions_for_usefulness',
                        'conditions_for_misleading'}
        found_keys = set(parsed.keys())
        missing = expected_keys - found_keys
        if missing:
            return parsed, False, f'Missing keys: {missing}'
        return parsed, True, ''
    except json.JSONDecodeError as e:
        return {}, False, f'JSONDecodeError: {str(e)[:100]}'


print('Response data structure defined.')

## Section 4: Per-provider call dispatchers

Each provider has its own SDK and quirks. Wrapping each into a uniform interface that returns `(raw_text, input_tokens, output_tokens, error_str)`.

In [ ]:
MAX_OUTPUT_TOKENS = 800  # generous for the 5-field JSON response
REQUEST_TIMEOUT = 60  # seconds

def call_anthropic(api_model: str, prompt: str) -> tuple:
    """Returns (raw_text, input_tokens, output_tokens, error_str)."""
    try:
        resp = anthropic_client.messages.create(
            model=api_model,
            max_tokens=MAX_OUTPUT_TOKENS,
            messages=[{'role': 'user', 'content': prompt}],
            timeout=REQUEST_TIMEOUT
        )
        return (
            resp.content[0].text,
            resp.usage.input_tokens,
            resp.usage.output_tokens,
            ''
        )
    except Exception as e:
        return ('', 0, 0, f'Anthropic error: {type(e).__name__}: {str(e)[:200]}')


def call_openai(api_model: str, prompt: str) -> tuple:
    try:
        resp = openai_client.chat.completions.create(
            model=api_model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=MAX_OUTPUT_TOKENS,
            timeout=REQUEST_TIMEOUT
        )
        return (
            resp.choices[0].message.content,
            resp.usage.prompt_tokens,
            resp.usage.completion_tokens,
            ''
        )
    except Exception as e:
        return ('', 0, 0, f'OpenAI error: {type(e).__name__}: {str(e)[:200]}')


def call_xai(api_model: str, prompt: str) -> tuple:
    try:
        resp = xai_client.chat.completions.create(
            model=api_model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=MAX_OUTPUT_TOKENS,
            timeout=REQUEST_TIMEOUT
        )
        return (
            resp.choices[0].message.content,
            resp.usage.prompt_tokens,
            resp.usage.completion_tokens,
            ''
        )
    except Exception as e:
        return ('', 0, 0, f'xAI error: {type(e).__name__}: {str(e)[:200]}')


def call_deepseek(api_model: str, prompt: str) -> tuple:
    try:
        resp = deepseek_client.chat.completions.create(
            model=api_model,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=MAX_OUTPUT_TOKENS,
            timeout=REQUEST_TIMEOUT
        )
        return (
            resp.choices[0].message.content,
            resp.usage.prompt_tokens,
            resp.usage.completion_tokens,
            ''
        )
    except Exception as e:
        return ('', 0, 0, f'DeepSeek error: {type(e).__name__}: {str(e)[:200]}')


def call_google(api_model: str, prompt: str) -> tuple:
    try:
        model = genai.GenerativeModel(api_model)
        resp = model.generate_content(
            prompt,
            generation_config={'max_output_tokens': MAX_OUTPUT_TOKENS},
            request_options={'timeout': REQUEST_TIMEOUT}
        )
        # Token counts via usage_metadata when available
        in_tok = getattr(resp.usage_metadata, 'prompt_token_count', 0) if hasattr(resp, 'usage_metadata') else 0
        out_tok = getattr(resp.usage_metadata, 'candidates_token_count', 0) if hasattr(resp, 'usage_metadata') else 0
        return (resp.text, in_tok, out_tok, '')
    except Exception as e:
        return ('', 0, 0, f'Google error: {type(e).__name__}: {str(e)[:200]}')


PROVIDER_DISPATCH = {
    'anthropic': call_anthropic,
    'openai': call_openai,
    'xai': call_xai,
    'deepseek': call_deepseek,
    'google': call_google,
}

print('Provider dispatchers defined for:', list(PROVIDER_DISPATCH.keys()))

## Section 5: Select pilot unit and filter execution plan

In [ ]:
# Choose pilot unit. Default: first C8 unit (synthetic CBT, clear signal).
# Override by setting PILOT_UNIT_ID to any unit_id from the sample.
PILOT_UNIT_ID = None  # set to a specific unit_id to override default

if PILOT_UNIT_ID is None:
    # Pick first C8 unit
    c8_calls = [c for c in EXECUTION_PLAN if c['text_id'] == 'C8']
    PILOT_UNIT_ID = c8_calls[0]['unit_id']

pilot_calls = [c for c in EXECUTION_PLAN if c['unit_id'] == PILOT_UNIT_ID]
pilot_unit_text = pilot_calls[0]['unit_text']

print(f'Pilot unit: {PILOT_UNIT_ID}')
print(f'Pilot text: {pilot_unit_text}')
print(f'Pilot calls: {len(pilot_calls)} (expected 40: 1 unit × 8 frames × 5 models)')

# Breakdown
from collections import Counter
by_frame = Counter(c['frame_id'] for c in pilot_calls)
by_model = Counter(c['model_id'] for c in pilot_calls)
print(f'\nBy frame: {dict(by_frame)}')
print(f'By model: {dict(by_model)}')

## Section 6: Execute pilot calls

Runs all 40 calls sequentially with brief pacing between requests. Stores each response immediately so a mid-execution failure doesn't lose completed work.

In [ ]:
PACING_SECONDS = 0.5  # brief pause between calls to avoid rate limits

pilot_responses = []
pilot_start = time.time()

for i, call in enumerate(pilot_calls, 1):
    print(f'[{i:>2}/{len(pilot_calls)}] {call["frame_id"]:24} | {call["model_id"]:24} ... ', end='', flush=True)
    
    dispatcher = PROVIDER_DISPATCH[call['provider']]
    call_start = time.time()
    raw, in_tok, out_tok, error = dispatcher(call['api_model'], call['rendered_prompt'])
    wall_clock = time.time() - call_start
    
    # Parse the response
    parsed, parse_ok, parse_err = extract_json_from_response(raw) if raw else ({}, False, 'no response')
    
    response = CouncilResponse(
        call_id=call['call_id'],
        unit_id=call['unit_id'],
        text_id=call['text_id'],
        frame_id=call['frame_id'],
        frame_version=call['frame_version'],
        prompt_sha256=call['prompt_sha256'],
        model_id=call['model_id'],
        provider=call['provider'],
        api_model=call['api_model'],
        timestamp=datetime.now(timezone.utc).isoformat(),
        wall_clock_seconds=wall_clock,
        raw_response=raw,
        parsed_json=parsed,
        parse_succeeded=parse_ok,
        parse_error=parse_err,
        input_tokens=in_tok,
        output_tokens=out_tok,
        error=error,
        retry_count=0
    )
    pilot_responses.append(response)
    
    # Print status
    if error:
        print(f'ERROR ({wall_clock:.1f}s)')
    elif not parse_ok:
        print(f'PARSE-FAIL ({wall_clock:.1f}s)')
    else:
        print(f'OK ({wall_clock:.1f}s, {in_tok}+{out_tok} tok)')
    
    # Save immediately so we don't lose progress on crash
    with open(f'{PROJECT_ROOT}/stage5/responses/{call["call_id"]}.json', 'w', encoding='utf-8') as f:
        json.dump(asdict(response), f, indent=2, ensure_ascii=False)
    
    time.sleep(PACING_SECONDS)

pilot_duration = time.time() - pilot_start
print(f'\nPilot complete in {pilot_duration:.1f}s ({pilot_duration/60:.1f} min)')

## Section 7: Pilot validation report

In [ ]:
# Compute pass/fail against each success criterion
total = len(pilot_responses)
errors = [r for r in pilot_responses if r.error]
no_response = [r for r in pilot_responses if not r.raw_response and not r.error]
parse_failures = [r for r in pilot_responses if r.raw_response and not r.parse_succeeded]
successes = [r for r in pilot_responses if r.parse_succeeded]

print(f'\nPilot Validation Report')
print('=' * 60)
print(f'Total calls: {total}')
print(f'Successes (clean JSON parse): {len(successes)} ({100*len(successes)/total:.1f}%)')
print(f'Parse failures: {len(parse_failures)} ({100*len(parse_failures)/total:.1f}%)')
print(f'API errors: {len(errors)}')
print(f'No response: {len(no_response)}')

print(f'\nSuccess Criteria:')
print(f'  1. All API clients authenticate: ', end='')
providers_with_responses = set(r.provider for r in pilot_responses if r.raw_response)
providers_expected = set(r.provider for r in pilot_responses)
if providers_with_responses == providers_expected:
    print('PASS')
else:
    missing_providers = providers_expected - providers_with_responses
    print(f'FAIL (missing: {missing_providers})')

print(f'  2. All 40 calls return responses: ', end='')
if len(no_response) + len(errors) == 0:
    print('PASS')
else:
    print(f'FAIL ({len(errors) + len(no_response)} failed)')

print(f'  3. ≥90% parse cleanly: ', end='')
parse_rate = len(successes) / total
if parse_rate >= 0.9:
    print(f'PASS ({parse_rate*100:.1f}%)')
else:
    print(f'FAIL ({parse_rate*100:.1f}%)')

print(f'  4. Frame variation visible: (manual inspection below)')
print(f'  5. Storage handles all shapes: PASS (all responses written to disk)')

In [ ]:
# Show errors and parse failures in detail (for debugging)
if errors:
    print('\n--- API ERRORS ---')
    for r in errors:
        print(f'\n[{r.call_id}]')
        print(f'  Provider: {r.provider} | Model: {r.api_model}')
        print(f'  Error: {r.error}')

if parse_failures:
    print('\n--- PARSE FAILURES ---')
    for r in parse_failures[:5]:  # show up to 5
        print(f'\n[{r.call_id}]')
        print(f'  Parse error: {r.parse_error}')
        print(f'  Raw response preview: {r.raw_response[:300]}')

In [ ]:
# Visual sanity check: how does the same model respond across frames?
# Pick one model and show all 8 frame responses
SANITY_MODEL = 'claude_sonnet_4_6'
model_responses = [r for r in pilot_responses if r.model_id == SANITY_MODEL]
model_responses.sort(key=lambda r: r.frame_id)

print(f'\nSanity check: {SANITY_MODEL} across all 8 frames')
print('=' * 70)
for r in model_responses:
    print(f'\n--- {r.frame_id} ---')
    if r.parse_succeeded:
        cf = r.parsed_json.get('communicative_function', '')
        print(f'communicative_function: {cf[:200] if cf else "(empty)"}')
    else:
        print(f'(parse failed: {r.parse_error})')
        print(f'raw: {r.raw_response[:200]}')

## Section 8: Cost analysis

In [ ]:
# Approximate per-million-token costs (as of mid-2026; verify with provider docs)
# Format: provider -> (input_per_M, output_per_M)
PRICING_USD = {
    'anthropic': (3.00, 15.00),    # claude-sonnet-4-6
    'openai': (2.50, 10.00),       # gpt-4o
    'google': (0.15, 0.60),        # gemini-2.5-flash
    'xai': (3.00, 15.00),          # grok-4 (estimate)
    'deepseek': (0.14, 0.28),      # deepseek-chat
}

total_cost = 0
cost_by_provider = {}
tokens_by_provider = {}

for r in pilot_responses:
    if r.provider not in PRICING_USD:
        continue
    in_price, out_price = PRICING_USD[r.provider]
    cost = (r.input_tokens / 1_000_000) * in_price + (r.output_tokens / 1_000_000) * out_price
    total_cost += cost
    cost_by_provider[r.provider] = cost_by_provider.get(r.provider, 0) + cost
    if r.provider not in tokens_by_provider:
        tokens_by_provider[r.provider] = {'in': 0, 'out': 0}
    tokens_by_provider[r.provider]['in'] += r.input_tokens
    tokens_by_provider[r.provider]['out'] += r.output_tokens

print(f'Pilot cost breakdown:')
print(f'{"provider":12} {"in_tokens":>10} {"out_tokens":>11} {"cost USD":>10}')
print('-' * 50)
for prov in sorted(cost_by_provider.keys()):
    tok = tokens_by_provider[prov]
    cost = cost_by_provider[prov]
    print(f'{prov:12} {tok["in"]:>10} {tok["out"]:>11} ${cost:>9.4f}')
print('-' * 50)
print(f'{"TOTAL":12} {"":>10} {"":>11} ${total_cost:>9.4f}')

print(f'\nExtrapolation to Stage 6 (50x scale): ~${total_cost * 50:.2f}')

## Section 9: Save pilot summary

In [ ]:
pilot_summary = {
    'stage': '5',
    'design_sha256': DESIGN_SHA,
    'pilot_unit_id': PILOT_UNIT_ID,
    'pilot_unit_text': pilot_unit_text,
    'execution_start': datetime.fromtimestamp(pilot_start, tz=timezone.utc).isoformat(),
    'execution_duration_seconds': pilot_duration,
    'total_calls': total,
    'successes': len(successes),
    'parse_failures': len(parse_failures),
    'api_errors': len(errors),
    'parse_rate': parse_rate,
    'pilot_cost_usd': round(total_cost, 4),
    'stage6_extrapolated_cost_usd': round(total_cost * 50, 2),
    'cost_by_provider': {p: round(c, 4) for p, c in cost_by_provider.items()},
    'tokens_by_provider': tokens_by_provider,
    'errors_detail': [
        {'call_id': r.call_id, 'provider': r.provider, 'model': r.api_model, 'error': r.error}
        for r in errors
    ],
    'parse_failures_detail': [
        {'call_id': r.call_id, 'parse_error': r.parse_error}
        for r in parse_failures
    ]
}

summary_path = f'{PROJECT_ROOT}/outputs/stage5_pilot_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(pilot_summary, f, indent=2, ensure_ascii=False)

print(f'Pilot summary saved: {summary_path}')
print(json.dumps(pilot_summary, indent=2))

## Section 10: Closeout and Stage 6 readiness

**Pilot outputs:**
- `stage5/responses/{call_id}.json` — individual response files (40 total)
- `stage5/responses/_index.json` — summary of all pilot responses (next cell builds this)
- `outputs/stage5_pilot_summary.json` — pilot validation report

**Decision point:**

If all success criteria passed and pilot cost extrapolates to within budget, Stage 6 is ready to run.

If any criterion failed:
- API authentication failures → fix Colab Secrets
- Parse failures → revisit `extract_json_from_response` or prompt wording
- Cost extrapolation over budget → adjust frame count or sample size before Stage 6
- No frame variation visible in responses → models may be ignoring framing, methodology concern

**What Stage 6 does (next):**
- Same code path as Stage 5
- Iterates over the full execution plan (2,000 calls instead of 40)
- Same storage structure
- Run unattended; check back in 3-6 hours

In [ ]:
# Build an index file for the pilot responses
index = {
    'pilot_unit_id': PILOT_UNIT_ID,
    'design_sha256': DESIGN_SHA,
    'responses': [
        {
            'call_id': r.call_id,
            'frame_id': r.frame_id,
            'model_id': r.model_id,
            'parse_succeeded': r.parse_succeeded,
            'has_error': bool(r.error),
            'wall_clock_seconds': r.wall_clock_seconds,
            'file': f'{r.call_id}.json'
        }
        for r in pilot_responses
    ]
}

with open(f'{PROJECT_ROOT}/stage5/responses/_index.json', 'w', encoding='utf-8') as f:
    json.dump(index, f, indent=2, ensure_ascii=False)

print('Pilot index saved.')
print(f'\nStage 5 status: {"READY FOR STAGE 6" if parse_rate >= 0.9 and len(errors) == 0 else "NEEDS FIXES BEFORE STAGE 6"}')